# Compare Timings of MCML and Surrogate Model

In [ ]:
import os
import pickle
import time
import warnings

import lightning as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly
import plotly.graph_objects as go
import torch
from omegaconf import DictConfig
from plotly.subplots import make_subplots
from rich.progress import track
from scipy.optimize import curve_fit

from mcmlnet.experiments.plotting import format_param_latex, setup_plot_style
from mcmlnet.experiments.utils import polynomial_func
from mcmlnet.training.data_loading.preprocessing import PreProcessor
from mcmlnet.training.models.base_model import BaseModel
from mcmlnet.training.models.kan_model import KANModel
from mcmlnet.utils.convenience import (
    load_trained_model,
    prepare_surrogate_model_data,
)
from mcmlnet.utils.load_configs import load_config
from mcmlnet.utils.mc_runner import RecomputeMC

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
if not os.path.exists(os.environ["plot_save_dir"]):
    os.makedirs(os.environ["plot_save_dir"])

## Define the Tissue Model

In [ ]:
# define base simulation repetition properties
N_SIMS = 1000
N_REPS = 30

# define re-used parameter values globally
a_ray = 0.0
cHb = 150.0  # mean value for (male) cHb
N_WVL = 351
N_RED_WVL = 15
wavelengths = np.linspace(300, 1000, num=N_WVL, endpoint=True, dtype=float) * 1e-9
# consistently subsample wavelengths (speed-up)
red_wavelengths = wavelengths[:: N_WVL // N_RED_WVL][:N_RED_WVL]


def generate_monte_carlo_data(n_samples: int = 10**6) -> pd.DataFrame:
    """
    Generate dummy Monte Carlo data based.

    Args:
        n_samples: Number of samples to generate.

    Returns:
        pd.DataFrame: DataFrame with the generated physiological parameter data.
    """
    random_state = np.random.RandomState(seed=42)
    base_layer = []
    for _i in range(3):
        base_layer.append(
            np.c_[
                random_state.uniform(0.001, 1.0, size=n_samples),
                random_state.uniform(0.001, 0.3, size=n_samples),
                random_state.uniform(500.0, 5000.0, size=n_samples),
                random_state.uniform(0.3, 3.0, size=n_samples),
                [a_ray] * n_samples,
                random_state.uniform(0.8, 0.95, size=n_samples),
                random_state.uniform(1.33, 1.54, size=n_samples),
                random_state.uniform(2e-5, 0.002, size=n_samples),
                [cHb] * n_samples,
            ]
        )
    # make last layer thick
    base_layer[2][:, 7] = 0.002 * 100
    columns = ["sao2", "vhb", "a_mie", "b_mie", "a_ray", "g", "n", "d", "chb"]
    # create parameter DataFrame
    header = RecomputeMC.generate_column_headers(columns, 3)
    physiological_data = pd.DataFrame(
        np.concatenate(base_layer, axis=1), columns=header
    )

    return physiological_data


# generate example Monte Carlo data
monte_carlo_data = generate_monte_carlo_data()
monte_carlo_data.head()

## Load Surrogate Models

In [ ]:
# load MLP and KAN model
cfg = load_config()
mlp_base = os.path.join(
    os.environ["data_dir"], cfg["surrogate"]["issi_model"]["base_path"]
)
mlp_checkpoint = cfg["surrogate"]["issi_model"]["checkpoint_path"]
kan_base = os.path.join(
    os.environ["data_dir"], cfg["surrogate"]["issi_kan_model"]["base_path"]
)
kan_checkpoint = cfg["surrogate"]["issi_kan_model"]["checkpoint_path"]

mlp_model, mlp_preprocessor, mlp_cfg = load_trained_model(
    mlp_base, mlp_checkpoint, BaseModel
)
kan_model, kan_preprocessor, kan_cfg = load_trained_model(
    kan_base, kan_checkpoint, KANModel
)

In [ ]:
# drop all columns with "chb" in their name
physiological_data = torch.from_numpy(
    monte_carlo_data.drop(
        columns=[col for col in monte_carlo_data.columns if "chb" in col]
    ).to_numpy()
).float()
print(f"Physiological data shape: {physiological_data.shape}")

# add dummy reflectance values
physiological_data = torch.cat(
    [physiological_data, torch.zeros(physiological_data.shape[0], N_RED_WVL)], dim=1
)

# use same subsampling as for MCML
np.random.seed(42)
random_indices = np.random.choice(
    len(monte_carlo_data) // N_SIMS, N_REPS, replace=False
)

## Run the Surrogate Models' Predictions

In [ ]:
def pre_process_data_and_run_surrogate_model(
    parameters: torch.Tensor,
    model: pl.LightningModule,
    preprocessor: PreProcessor,
    cfg: DictConfig,
    use_torch_compile: bool,
) -> tuple[list[float], torch.Tensor]:
    """
    Pre-process the data and run the surrogate model. As the MC sim.s do physio
    to physical conversion internally, include this overhead in the timing, as
    well as the model loading. This will cause one-time overhead, but is more
    realistic.

    Args:
        parameters: Tensor of shape (n_samples, n_parameters) containing
            the physiological parameters.
        model: The surrogate model to use.
        preprocessor: The preprocessor to use.
        cfg: The configuration to use.
        use_torch_compile: Whether to use torch.compile.

    Returns:
        A tuple containing the timings and the reflectances.
    """
    surrogate_timings = []
    reflectances = []

    for _i, val in track(enumerate(random_indices), total=N_REPS, transient=True):
        start = time.time()
        # apply normalization
        preprocessed_data = (
            prepare_surrogate_model_data(
                preprocessor,
                cfg,
                parameters[val * N_SIMS : (val + 1) * N_SIMS].clone(),
            )
            .float()
            .cuda()
        )

        # compile model.forward if desired
        if use_torch_compile:
            try:
                forward = torch.compile(model.forward)
            except Exception as err:
                warnings.warn(
                    f"torch.compile disabled "
                    f"({err.__class__.__name__}: {err}). "
                    "Running regular model.forward() instead.",
                    stacklevel=2,
                )
                forward = model
        else:
            forward = model

        # avoid VRAM memory issues by detaching and moving the result to CPU
        reflectance = forward(preprocessed_data[..., :-1]).detach().cpu()
        if cfg.preprocessing.log_intensity:
            reflectance = 10**reflectance

        surrogate_timings.append(time.time() - start)
        reflectances.append(reflectance)

    return surrogate_timings, torch.stack(reflectances, dim=0)

In [ ]:
# evaluate MLP model on validation set, individual reflectance values
surrogate_timings_mlp, _ = pre_process_data_and_run_surrogate_model(
    physiological_data, mlp_model, mlp_preprocessor, mlp_cfg, use_torch_compile=False
)
surrogate_timings_mlp_compiled, _ = pre_process_data_and_run_surrogate_model(
    physiological_data, mlp_model, mlp_preprocessor, mlp_cfg, use_torch_compile=True
)
surrogate_timings_kan, _ = pre_process_data_and_run_surrogate_model(
    physiological_data, kan_model, kan_preprocessor, kan_cfg, use_torch_compile=False
)
surrogate_timings_kan_compiled, _ = pre_process_data_and_run_surrogate_model(
    physiological_data, kan_model, kan_preprocessor, kan_cfg, use_torch_compile=True
)
# put all timings into a single list
surrogate_timings = [
    surrogate_timings_mlp,
    surrogate_timings_mlp_compiled,
    surrogate_timings_kan,
    surrogate_timings_kan_compiled,
]

## Run the MCML Simulation

In [ ]:
# run 100 different sets of 10 simulations with 1 mio photons each
mcml_timings = []

# Generate 10 sets of 100 random integers each
np.random.seed(42)
random_indices = np.random.choice(
    len(monte_carlo_data) // N_SIMS, N_REPS, replace=False
)

storage_dir = os.path.join(os.environ["cache_dir"], "timing_data")

try:
    with open(os.path.join(storage_dir, "mcml_timings.pkl"), "rb") as file:
        mcml_timings = pickle.load(file)
except FileNotFoundError:
    for _n_photons in [1e6, 5e6, 1e7, 1e8]:
        _sim_timings = []
        _reflectances = []
        for i, val in track(enumerate(random_indices), total=N_REPS, transient=True):
            start = time.time()
            _refl = RecomputeMC(
                red_wavelengths, nr_photons=_n_photons
            ).run_simulation_from_df(
                monte_carlo_data[val * N_SIMS : (val + 1) * N_SIMS],
                os.path.join(
                    storage_dir, f"timing_data_{_n_photons // 1e6}_mio_photons_{i}/"
                ),
                batch_size=N_SIMS,
            )
            _sim_timings.append(time.time() - start)
            _reflectances.append(_refl)

        # sanity check the results with the surrogate model predictions
        _, _surr_refl = pre_process_data_and_run_surrogate_model(
            physiological_data,
            mlp_model,
            mlp_preprocessor,
            mlp_cfg,
            use_torch_compile=False,
        )
        _surr_refl = _surr_refl.detach().cpu().numpy()
        _reflectances = np.stack(_reflectances, axis=0)
        if not np.allclose(_reflectances, _surr_refl, rtol=1e-2, atol=1e-3):
            print(
                "Warning: MCML simulation results and surrogate model predictions "
                f"do not match for set {i}."
            )
            diff = np.where(
                np.abs(_reflectances - _surr_refl) / _reflectances > 1e-2
            ) or np.where(np.abs(_reflectances - _surr_refl) > 1e-3)
            print(f"MCML reflectance: {_reflectances[diff]}")
            print(f"Surrogate reflectance: {_surr_refl[diff]}")

        mcml_timings.append(_sim_timings)

    with open(os.path.join(storage_dir, "mcml_timings.pkl"), "wb") as f:
        pickle.dump(mcml_timings, f)

print(f"MCML timings: {mcml_timings}")

## Plot the Timings

In [ ]:
labels = ["MLP", "MLP Compiled", "KAN", "KAN Compiled"]

plt.figure(figsize=(10, 6))
for i, timings in enumerate(surrogate_timings):
    plt.plot(timings, label=f"{labels[i]}")
    plt.title("Surrogate Model Runtime")
plt.xlabel("Simulation Subset")
plt.ylabel("Runtime (s)")
plt.yscale("log")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Define labels and colors
n_photons = [1e6, 5e6, 1e7, 1e8]
colors = [
    "red",
    "firebrick",
    "maroon",
    "black",
]

fig = go.Figure()

for i, timings in enumerate(mcml_timings):
    fig.add_trace(
        go.Violin(
            y=timings,
            name=f"GPUMCML ({int(n_photons[i] // 1e6)}M Photons)",
            box_visible=True,
            line_color=colors[i],
            marker={"size": 2},
            side="negative",
            points="all",
            pointpos=0.4,
        ),
    )
for i, timings in enumerate(surrogate_timings):
    if i > 0:
        break
    fig.add_trace(
        go.Violin(
            y=timings[1:],
            name="Our Surrogate Model",
            box_visible=True,
            line_color=plotly.colors.qualitative.T10[0],
            marker={"size": 2},
            side="negative",
            points="all",
            pointpos=0.4,
        ),
    )
fig.update_layout(
    title="MCML vs. Surrogate Model Inference Time",
    hovermode="x unified",
    yaxis={
        "title": "Inference Time [s]",
        "type": "log",
        "range": [-3, 4],
    },
    legend={
        "title": "Inference Method",
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "center",
        "x": 0.5,
    },
    violingap=0,
    width=400,
    height=500,
    template="presentation",
    font={
        "family": "Times New Roman",
        "size": 10,
        "color": "black",
    },
)
fig.write_image(
    os.path.join(
        os.getenv("plot_save_dir"), "mcml_vs_surrogate_model_inference_time.pdf"
    ),
    engine="kaleido",
)
fig.show()

# print average and median timings for each method
for i, timings in enumerate(surrogate_timings):
    print(f"{labels[i]} Average Timing: {np.mean(timings):.6f} seconds")
    print(f"{labels[i]} Median Timing: {np.median(timings):.6f} seconds")
for i, timings in enumerate(mcml_timings):
    print(
        f"MCML {int(n_photons[i] // 1e6)}M Average Timing: "
        f"{np.mean(timings):.6f} seconds"
    )
    print(
        f"MCML {int(n_photons[i] // 1e6)}M Median Timing: "
        f"{np.median(timings):.6f} seconds"
    )

In [ ]:
fig.write_html(
    str(os.getenv("plot_save_dir")) + "timing_comparison_MCML_surrogate.html"
)
fig.write_image(
    str(os.getenv("plot_save_dir")) + "timing_comparison_MCML_surrogate.pdf",
    width=400,
    height=600,
)

## Check the Timings of Different Batch Sizes

In [ ]:
simulation_batch_sizes = [10, 100, 1000, 10000, 100000]

result_data = {}

for n_sims in simulation_batch_sizes:
    surrogate_timings = []

    try:
        random_indices = np.random.choice(
            len(monte_carlo_data) // n_sims, N_REPS, replace=False
        )
    except ValueError:
        random_indices = np.random.choice(
            len(monte_carlo_data) // n_sims, N_REPS, replace=True
        )

    for _i, val in track(enumerate(random_indices), total=N_REPS, transient=True):
        start = time.time()

        # apply normalization
        preprocessed_data = prepare_surrogate_model_data(
            mlp_preprocessor,
            mlp_cfg,
            physiological_data[val * n_sims : (val + 1) * n_sims].clone(),
        )

        reflectance = (
            mlp_model(preprocessed_data[..., :-1].float().cuda()).detach().cpu()
        )
        if mlp_cfg.preprocessing.log_intensity:
            reflectance = 10**reflectance

        surrogate_timings.append(time.time() - start)
    result_data[n_sims] = surrogate_timings

result_df = pd.DataFrame(result_data)
result_df.head()

In [ ]:
# plot the results
plt.figure(figsize=(10, 6))
for col in result_df.columns:
    plt.plot(result_df[col], label=f"Batch size {col}")
plt.title("Surrogate Model Timings for Different Batch Sizes")
plt.xlabel("Simulation Subset")
plt.ylabel("Runtime [s]")
plt.yscale("log")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

# plot the results with Plotly
fig = make_subplots(rows=1, cols=1)
for col in result_df.columns:
    fig.add_trace(
        go.Violin(
            y=result_df[col],
            name=f"Batch size {col}",
            box_visible=True,
            side="negative",
            points="all",
            pointpos=0.4,
        ),
        row=1,
        col=1,
    )
fig.update_yaxes(
    title="Time [s]",
    hoverformat=".2e",
    type="log",
)
fig.update_layout(
    title="Surrogate Model Timings for Different Batch Sizes",
    hovermode="x unified",
    yaxis={"title": "Time [s]", "hoverformat": ".2e", "type": "log"},
    legend={
        "title": "Batch Size",
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "center",
        "x": 0.5,
    },
    width=800,
    height=400,
    template="presentation",
    font={
        "family": "Times New Roman",
        "size": 10,
        "color": "black",
    },
)
fig.write_image(
    os.path.join(
        os.getenv("plot_save_dir"),
        "surrogate_model_inference_time_multiple_batch_sizes.pdf",
    )
)
fig.show()

### Fit a Power-Law to the Batch Size Time Scaling

In [ ]:
# Flatten the timings and batch sizes
all_timings = []
all_batch_sizes = []

for col in result_df.columns:
    all_timings.extend(result_df[col].tolist())
    all_batch_sizes.extend([col] * len(result_df[col]))

# Convert to numpy arrays
all_timings = np.array(all_timings)
all_batch_sizes = np.array(all_batch_sizes)

In [ ]:
# Fit the power law to the data
popt, pcov = curve_fit(polynomial_func, all_batch_sizes, all_timings)
a, b, c = popt  # Extract fitted parameters
da, db, dc = np.sqrt(np.diag(pcov))  # Extract uncertainties

# Generate fitted curve
batch_sizes_fit = np.linspace(
    min(simulation_batch_sizes), max(simulation_batch_sizes), 100
)
timings_fit = polynomial_func(batch_sizes_fit, a, b, c)

In [ ]:
setup_plot_style()


# Plot the data and fitted curve
plt.figure(figsize=(6.5, 4))
for col in result_df.columns:
    plt.scatter([int(col)] * len(result_df[col]), result_df[col], alpha=0.5, marker="x")
plt.plot(
    batch_sizes_fit,
    timings_fit,
    label=(
        rf"Fitted Power Law: $t$ = {format_param_latex(a)}s $\cdot$ Batch Size"
        rf"$^{{{b:.2f}}} + ${format_param_latex(c)}s"
    ),
    color="red",
)
plt.fill_between(
    batch_sizes_fit,
    polynomial_func(batch_sizes_fit, a - da, b - db, c - dc),
    polynomial_func(batch_sizes_fit, a + da, b + db, c + dc),
    color="red",
    alpha=0.2,
    label="Fitting Parameter Standard Deviation Uncertainty Band",
)
plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"Batch Size $B$ [$\times$ 15 Wavelengths]")
plt.ylabel("Time [s]")
# plt.title("Surrogate Model Timings for Different Batch Sizes")
plt.legend()
plt.grid()
plt.savefig(
    os.path.join(
        os.getenv("plot_save_dir"),
        "surrogate_model_inference_time_multiple_batch_sizes_fit.pdf",
    )
)
plt.show()

# Print fitted parameters
print(f"Fitted parameters: a = {a:.4e}, b = {b:.4f}, c = {c:.4e}")
print(f"Uncertainties: da = {da:.4e}, db = {db:.4f}, dc = {dc:.4e}")